# 05 - Run Experiments

5x2 cross-validation for the linear, feedforward, and autoencoder networks on each data set.

In [2]:
import pandas as pd
import numpy as np

import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.preprocessing import load_processed, prepare_xy
from src.neural_network import NeuralNetwork
from src.linear_network import LogisticRegression, LinearRegression
from src.autoencoder import AutoencoderPredictor
from src.evaluation import cross_validation_loop
from src.tuning import holdout_split, random_search

In [3]:
seed = 0

### Breast Cancer Dataset

In [3]:
# Breast Cancer Dataset
data, meta = load_processed("../data/processed/breast_cancer_processed.csv")
X, y, y_labels = prepare_xy(data, meta)

# 20% tuning set, 80% for the 5x2 CV (same split for every model)
cv_idx, val_idx = holdout_split(len(X), y_labels, 0.2, seed=seed)
X_cv, y_cv, yl_cv = X[cv_idx], y[cv_idx], y_labels[cv_idx]
X_val, y_val, yl_val = X[val_idx], y[val_idx], y_labels[val_idx]

In [4]:
# ---------------- Logistic Regression ----------------
print("Logistic Regression")
log_space = {"learning_rate": ("log_uniform", 0.001, 1.0), "batch_size": ("int", 8, max(8, X_cv.shape[0]//10))}
log_factory = lambda c, s: LogisticRegression(X.shape[1], y.shape[1], c["learning_rate"], seed=s)

best_log, log_trials = random_search(
    log_factory, log_space, X_cv, y_cv, X_val, y_val, yl_val, mode="classification",
    n_trials=30, max_epochs=1000, checkpoint_every=50, seed=seed,
    log_path="../results/tuning/breast_cancer_logistic.csv")
print("Best:", best_log)

results = cross_validation_loop(
    X_cv, y_cv, yl_cv, lambda: log_factory(best_log, seed),
    epochs=best_log["epochs"], batch_size=best_log["batch_size"], seed=seed)
print(f"Mean Classification Error: {np.mean(results['error'])} \nMean Cross Entropy Loss: {np.mean(results['cross_entropy'])}\n")

Logistic Regression
trial   1/30  val_loss=0.0965  epochs=50  {'learning_rate': 0.081448869730761, 'batch_size': 32}
trial   2/30  val_loss=0.0967  epochs=1000  {'learning_rate': 0.0013271517125308802, 'batch_size': 20}
trial   3/30  val_loss=0.0966  epochs=1000  {'learning_rate': 0.0011209413928285786, 'batch_size': 16}
trial   4/30  val_loss=0.0969  epochs=750  {'learning_rate': 0.5473520283788641, 'batch_size': 47}
trial   5/30  val_loss=0.0963  epochs=100  {'learning_rate': 0.06605524798050313, 'batch_size': 54}
trial   6/30  val_loss=0.0964  epochs=100  {'learning_rate': 0.04274398161693902, 'batch_size': 43}
trial   7/30  val_loss=0.0964  epochs=150  {'learning_rate': 0.6385828802382952, 'batch_size': 21}
trial   8/30  val_loss=0.1040  epochs=1000  {'learning_rate': 0.0010190969469382445, 'batch_size': 47}
trial   9/30  val_loss=0.0968  epochs=50  {'learning_rate': 0.37343311708676946, 'batch_size': 34}
trial  10/30  val_loss=0.0968  epochs=900  {'learning_rate': 0.15451346766855

In [5]:
# ---------------- Neural Network ----------------
print("Neural Network")
nn_space = {"hidden1": ("int", 2, min(X.shape[1]*2, 32)), "hidden2": ("int", 2, min(X.shape[1]*2, 32)),
            "learning_rate": ("log_uniform", 0.001, 1.0), "batch_size": ("int", 8, max(8, X_cv.shape[0]//10))}
nn_factory = lambda c, s: NeuralNetwork(
    [X.shape[1], c["hidden1"], c["hidden2"], y.shape[1]],
    mode="classification", learning_rate=c["learning_rate"], seed=s)

best_nn, nn_trials = random_search(
    nn_factory, nn_space, X_cv, y_cv, X_val, y_val, yl_val, mode="classification",
    n_trials=60, max_epochs=1000, checkpoint_every=50,
    constraint=lambda c: c["hidden2"] <= c["hidden1"], seed=seed,
    log_path="../results/tuning/breast_cancer_nn.csv")
print("Best:", best_nn)

results = cross_validation_loop(
    X_cv, y_cv, yl_cv, lambda: nn_factory(best_nn, seed),
    epochs=best_nn["epochs"], batch_size=best_nn["batch_size"], seed=seed)
print(f"Mean Classification Error: {np.mean(results['error'])} \nMean Cross Entropy Loss: {np.mean(results['cross_entropy'])}")

Neural Network
trial   1/60  val_loss=0.5135  epochs=1000  {'hidden1': 16, 'hidden2': 12, 'learning_rate': 0.006447036682716461, 'batch_size': 22}
trial   2/60  val_loss=0.1065  epochs=50  {'hidden1': 14, 'hidden2': 12, 'learning_rate': 0.6385828802382952, 'batch_size': 34}
trial   3/60  val_loss=0.1030  epochs=50  {'hidden1': 16, 'hidden2': 11, 'learning_rate': 0.15451346766855226, 'batch_size': 9}
trial   4/60  val_loss=0.1391  epochs=50  {'hidden1': 16, 'hidden2': 4, 'learning_rate': 0.3886303977238022, 'batch_size': 9}
trial   5/60  val_loss=0.1017  epochs=500  {'hidden1': 11, 'hidden2': 3, 'learning_rate': 0.018537948170285826, 'batch_size': 22}
trial   6/60  val_loss=0.5743  epochs=1000  {'hidden1': 8, 'hidden2': 2, 'learning_rate': 0.002359662167270631, 'batch_size': 8}
trial   7/60  val_loss=0.0971  epochs=250  {'hidden1': 13, 'hidden2': 10, 'learning_rate': 0.07017062333532993, 'batch_size': 39}
trial   8/60  val_loss=0.1112  epochs=50  {'hidden1': 14, 'hidden2': 8, 'learning_

In [6]:
# ---------------- Autoencoder Neural Network ----------------
print("Autoencoder Neural Network")
ae_space = {
    "n_encoding": ("int", 2, X.shape[1] - 1),                       
    "hidden2": ("int", 2, min(X.shape[1] * 2, 32)),
    "learning_rate": ("log_uniform", 0.001, 1.0),
    "batch_size": ("int", 8, max(8, X_cv.shape[0] // 10))             
}
ae_factory = lambda c, s: AutoencoderPredictor(
    X.shape[1], c["n_encoding"], c["hidden2"], y.shape[1],
    mode="classification", learning_rate=c["learning_rate"], seed=s)

best_ae, ae_trials = random_search(
    ae_factory, ae_space, X_cv, y_cv, X_val, y_val, yl_val, mode="classification",
    n_trials=60, max_epochs=1000, checkpoint_every=50, seed=seed,
    log_path="../results/tuning/breast_cancer_autoencoder.csv")
print("Best:", best_ae)

results = cross_validation_loop(
    X_cv, y_cv, yl_cv, lambda: ae_factory(best_ae, seed),
    epochs=best_ae["epochs"], batch_size=best_ae["batch_size"], seed=seed)
print(f"Mean Classification Error: {np.mean(results['error'])} \nMean Cross Entropy Loss: {np.mean(results['cross_entropy'])}")

Autoencoder Neural Network
trial   1/60  val_loss=0.1051  epochs=1000  {'n_encoding': 7, 'hidden2': 12, 'learning_rate': 0.006447036682716461, 'batch_size': 22}
trial   2/60  val_loss=0.0997  epochs=600  {'n_encoding': 2, 'hidden2': 3, 'learning_rate': 0.2753028565072983, 'batch_size': 8}
trial   3/60  val_loss=0.0957  epochs=1000  {'n_encoding': 6, 'hidden2': 17, 'learning_rate': 0.06605524798050313, 'batch_size': 54}
trial   4/60  val_loss=0.0907  epochs=700  {'n_encoding': 7, 'hidden2': 12, 'learning_rate': 0.6385828802382952, 'batch_size': 34}
trial   5/60  val_loss=0.6213  epochs=1000  {'n_encoding': 3, 'hidden2': 15, 'learning_rate': 0.0010190969469382445, 'batch_size': 26}
trial   6/60  val_loss=0.0946  epochs=400  {'n_encoding': 8, 'hidden2': 11, 'learning_rate': 0.15451346766855226, 'batch_size': 9}
trial   7/60  val_loss=0.0990  epochs=350  {'n_encoding': 7, 'hidden2': 4, 'learning_rate': 0.3886303977238022, 'batch_size': 9}
trial   8/60  val_loss=0.0990  epochs=1000  {'n_enc

### Car Evaluation Dataset

In [4]:
# Car Evaluation Dataset
data, meta = load_processed("../data/processed/car_evaluation_processed.csv")
X, y, y_labels = prepare_xy(data, meta)

# 20% tuning set, 80% for the 5x2 CV (same split for every model)
cv_idx, val_idx = holdout_split(len(X), y_labels, 0.2, seed=seed)
X_cv, y_cv, yl_cv = X[cv_idx], y[cv_idx], y_labels[cv_idx]
X_val, y_val, yl_val = X[val_idx], y[val_idx], y_labels[val_idx]

In [5]:
# ---------------- Logistic Regression ----------------
print("Logistic Regression")
log_space = {"learning_rate": ("log_uniform", 0.001, 1.0), "batch_size": ("int", 8, max(8, X_cv.shape[0]//10))}
log_factory = lambda c, s: LogisticRegression(X.shape[1], y.shape[1], c["learning_rate"], seed=s)

best_log, log_trials = random_search(
    log_factory, log_space, X_cv, y_cv, X_val, y_val, yl_val, mode="classification",
    n_trials=30, max_epochs=1000, checkpoint_every=50, seed=seed,
    log_path="../results/tuning/car_evaluation_logistic.csv")
print("Best:", best_log)

results = cross_validation_loop(
    X_cv, y_cv, yl_cv, lambda: log_factory(best_log, seed),
    epochs=best_log["epochs"], batch_size=best_log["batch_size"], seed=seed)
print(f"Mean Classification Error: {np.mean(results['error'])} \nMean Cross Entropy Loss: {np.mean(results['cross_entropy'])}\n")

Logistic Regression
trial   1/30  val_loss=0.3932  epochs=950  {'learning_rate': 0.081448869730761, 'batch_size': 74}
trial   2/30  val_loss=0.4749  epochs=1000  {'learning_rate': 0.0013271517125308802, 'batch_size': 43}
trial   3/30  val_loss=0.4660  epochs=1000  {'learning_rate': 0.0011209413928285786, 'batch_size': 30}
trial   4/30  val_loss=0.3921  epochs=200  {'learning_rate': 0.5473520283788641, 'batch_size': 114}
trial   5/30  val_loss=0.3979  epochs=950  {'learning_rate': 0.06605524798050313, 'batch_size': 135}
trial   6/30  val_loss=0.3999  epochs=1000  {'learning_rate': 0.04274398161693902, 'batch_size': 103}
trial   7/30  val_loss=0.3914  epochs=150  {'learning_rate': 0.6385828802382952, 'batch_size': 44}
trial   8/30  val_loss=0.5484  epochs=1000  {'learning_rate': 0.0010190969469382445, 'batch_size': 114}
trial   9/30  val_loss=0.3916  epochs=400  {'learning_rate': 0.37343311708676946, 'batch_size': 80}
trial  10/30  val_loss=0.3935  epochs=100  {'learning_rate': 0.1545134

In [6]:
# ---------------- Neural Network ----------------
print("Neural Network")
nn_space = {"hidden1": ("int", 2, min(X.shape[1]*2, 32)), "hidden2": ("int", 2, min(X.shape[1]*2, 32)),
            "learning_rate": ("log_uniform", 0.001, 1.0), "batch_size": ("int", 8, max(8, X_cv.shape[0]//10))}
nn_factory = lambda c, s: NeuralNetwork(
    [X.shape[1], c["hidden1"], c["hidden2"], y.shape[1]],
    mode="classification", learning_rate=c["learning_rate"], seed=s)

best_nn, nn_trials = random_search(
    nn_factory, nn_space, X_cv, y_cv, X_val, y_val, yl_val, mode="classification",
    n_trials=60, max_epochs=1000, checkpoint_every=50,
    constraint=lambda c: c["hidden2"] <= c["hidden1"], seed=seed,
    log_path="../results/tuning/car_evaluation_nn.csv")
print("Best:", best_nn)

results = cross_validation_loop(
    X_cv, y_cv, yl_cv, lambda: nn_factory(best_nn, seed),
    epochs=best_nn["epochs"], batch_size=best_nn["batch_size"], seed=seed)
print(f"Mean Classification Error: {np.mean(results['error'])} \nMean Cross Entropy Loss: {np.mean(results['cross_entropy'])}")

Neural Network
trial   1/60  val_loss=0.8110  epochs=1000  {'hidden1': 11, 'hidden2': 9, 'learning_rate': 0.006447036682716461, 'batch_size': 48}
trial   2/60  val_loss=0.2763  epochs=450  {'hidden1': 2, 'hidden2': 2, 'learning_rate': 0.2753028565072983, 'batch_size': 10}
trial   3/60  val_loss=0.0324  epochs=450  {'hidden1': 10, 'hidden2': 8, 'learning_rate': 0.6385828802382952, 'batch_size': 79}
trial   4/60  val_loss=0.0353  epochs=900  {'hidden1': 11, 'hidden2': 8, 'learning_rate': 0.15451346766855226, 'batch_size': 12}
trial   5/60  val_loss=0.0042  epochs=1000  {'hidden1': 11, 'hidden2': 3, 'learning_rate': 0.3886303977238022, 'batch_size': 10}
trial   6/60  val_loss=0.4672  epochs=1000  {'hidden1': 7, 'hidden2': 2, 'learning_rate': 0.018537948170285826, 'batch_size': 47}
trial   7/60  val_loss=0.5078  epochs=1000  {'hidden1': 6, 'hidden2': 2, 'learning_rate': 0.002359662167270631, 'batch_size': 9}
trial   8/60  val_loss=0.2298  epochs=1000  {'hidden1': 9, 'hidden2': 7, 'learning

In [7]:
# ---------------- Autoencoder Neural Network ----------------
print("Autoencoder Neural Network")
ae_space = {
    "n_encoding": ("int", 2, X.shape[1] - 1),                       
    "hidden2": ("int", 2, min(X.shape[1] * 2, 32)),
    "learning_rate": ("log_uniform", 0.001, 1.0),
    "batch_size": ("int", 8, max(8, X_cv.shape[0] // 10))             
}
ae_factory = lambda c, s: AutoencoderPredictor(
    X.shape[1], c["n_encoding"], c["hidden2"], y.shape[1],
    mode="classification", learning_rate=c["learning_rate"], seed=s)

best_ae, ae_trials = random_search(
    ae_factory, ae_space, X_cv, y_cv, X_val, y_val, yl_val, mode="classification",
    n_trials=60, max_epochs=1000, checkpoint_every=50, seed=seed,
    log_path="../results/tuning/car_evaluation_autoencoder.csv")
print("Best:", best_ae)

results = cross_validation_loop(
    X_cv, y_cv, yl_cv, lambda: ae_factory(best_ae, seed),
    epochs=best_ae["epochs"], batch_size=best_ae["batch_size"], seed=seed)
print(f"Mean Classification Error: {np.mean(results['error'])} \nMean Cross Entropy Loss: {np.mean(results['cross_entropy'])}")

Autoencoder Neural Network
trial   1/60  val_loss=0.6697  epochs=1000  {'n_encoding': 5, 'hidden2': 9, 'learning_rate': 0.006447036682716461, 'batch_size': 48}
trial   2/60  val_loss=0.7052  epochs=800  {'n_encoding': 2, 'hidden2': 2, 'learning_rate': 0.2753028565072983, 'batch_size': 10}
trial   3/60  val_loss=0.5815  epochs=1000  {'n_encoding': 4, 'hidden2': 12, 'learning_rate': 0.06605524798050313, 'batch_size': 135}
trial   4/60  val_loss=0.3359  epochs=1000  {'n_encoding': 4, 'hidden2': 8, 'learning_rate': 0.6385828802382952, 'batch_size': 79}
trial   5/60  val_loss=0.8371  epochs=1000  {'n_encoding': 3, 'hidden2': 10, 'learning_rate': 0.0010190969469382445, 'batch_size': 59}
trial   6/60  val_loss=0.2865  epochs=950  {'n_encoding': 5, 'hidden2': 8, 'learning_rate': 0.15451346766855226, 'batch_size': 12}
trial   7/60  val_loss=0.3282  epochs=500  {'n_encoding': 5, 'hidden2': 3, 'learning_rate': 0.3886303977238022, 'batch_size': 10}
trial   8/60  val_loss=0.5569  epochs=1000  {'n_e

### House Votes Dataset

In [27]:
# House Votes Dataset
data, meta = load_processed("../data/processed/house_votes_processed.csv")
X, y, y_labels = prepare_xy(data, meta)

# 20% tuning set, 80% for the 5x2 CV (same split for every model)
cv_idx, val_idx = holdout_split(len(X), y_labels, 0.2, seed=seed)
X_cv, y_cv, yl_cv = X[cv_idx], y[cv_idx], y_labels[cv_idx]
X_val, y_val, yl_val = X[val_idx], y[val_idx], y_labels[val_idx]

In [9]:
# ---------------- Logistic Regression ----------------
print("Logistic Regression")
log_space = {"learning_rate": ("log_uniform", 0.001, 1.0), "batch_size": ("int", 8, max(8, X_cv.shape[0]//10))}
log_factory = lambda c, s: LogisticRegression(X.shape[1], y.shape[1], c["learning_rate"], seed=s)

best_log, log_trials = random_search(
    log_factory, log_space, X_cv, y_cv, X_val, y_val, yl_val, mode="classification",
    n_trials=30, max_epochs=1000, checkpoint_every=50, seed=seed,
    log_path="../results/tuning/house_votes_logistic.csv")
print("Best:", best_log)

results = cross_validation_loop(
    X_cv, y_cv, yl_cv, lambda: log_factory(best_log, seed),
    epochs=best_log["epochs"], batch_size=best_log["batch_size"], seed=seed)
print(f"Mean Classification Error: {np.mean(results['error'])} \nMean Cross Entropy Loss: {np.mean(results['cross_entropy'])}\n")

Logistic Regression
trial   1/30  val_loss=0.1007  epochs=150  {'learning_rate': 0.081448869730761, 'batch_size': 21}
trial   2/30  val_loss=0.1234  epochs=1000  {'learning_rate': 0.0013271517125308802, 'batch_size': 15}
trial   3/30  val_loss=0.1227  epochs=1000  {'learning_rate': 0.0011209413928285786, 'batch_size': 12}
trial   4/30  val_loss=0.0996  epochs=50  {'learning_rate': 0.5473520283788641, 'batch_size': 29}
trial   5/30  val_loss=0.1001  epochs=250  {'learning_rate': 0.06605524798050313, 'batch_size': 34}
trial   6/30  val_loss=0.1006  epochs=300  {'learning_rate': 0.04274398161693902, 'batch_size': 27}
trial   7/30  val_loss=0.1083  epochs=50  {'learning_rate': 0.6385828802382952, 'batch_size': 15}
trial   8/30  val_loss=0.1625  epochs=1000  {'learning_rate': 0.0010190969469382445, 'batch_size': 30}
trial   9/30  val_loss=0.1039  epochs=50  {'learning_rate': 0.37343311708676946, 'batch_size': 22}
trial  10/30  val_loss=0.1073  epochs=50  {'learning_rate': 0.1545134676685522

In [30]:
# ---------------- Neural Network ----------------
print("Neural Network")
nn_space = {"hidden1": ("int", 2, min(X.shape[1]*2, 32)), "hidden2": ("int", 2, min(X.shape[1]*2, 32)),
            "learning_rate": ("log_uniform", 0.001, 1.0), "batch_size": ("int", 8, max(8, X_cv.shape[0]//10))}
nn_factory = lambda c, s: NeuralNetwork(
    [X.shape[1], c["hidden1"], c["hidden2"], y.shape[1]],
    mode="classification", learning_rate=c["learning_rate"], seed=s)

best_nn, nn_trials = random_search(
    nn_factory, nn_space, X_cv, y_cv, X_val, y_val, yl_val, mode="classification",
    n_trials=60, max_epochs=1000, checkpoint_every=50,
    constraint=lambda c: c["hidden2"] <= c["hidden1"], seed=1,
    log_path="../results/tuning/house_votes_nn.csv")
print("Best:", best_nn)

results = cross_validation_loop(
    X_cv, y_cv, yl_cv, lambda: nn_factory(best_nn, seed),
    epochs=best_nn["epochs"], batch_size=best_nn["batch_size"], seed=1)
print(f"Mean Classification Error: {np.mean(results['error'])} \nMean Cross Entropy Loss: {np.mean(results['cross_entropy'])}")

Neural Network
trial   1/60  val_loss=0.0999  epochs=50  {'hidden1': 28, 'hidden2': 15, 'learning_rate': 0.3041639793970099, 'batch_size': 14}
trial   2/60  val_loss=0.1020  epochs=800  {'hidden1': 28, 'hidden2': 25, 'learning_rate': 0.041155694962153225, 'batch_size': 30}
trial   3/60  val_loss=0.1216  epochs=50  {'hidden1': 17, 'hidden2': 5, 'learning_rate': 0.7673112770284728, 'batch_size': 34}
trial   4/60  val_loss=0.1045  epochs=400  {'hidden1': 21, 'hidden2': 16, 'learning_rate': 0.06902555434658413, 'batch_size': 28}
trial   5/60  val_loss=0.1064  epochs=1000  {'hidden1': 20, 'hidden2': 17, 'learning_rate': 0.034093112644222705, 'batch_size': 34}
trial   6/60  val_loss=0.1329  epochs=50  {'hidden1': 25, 'hidden2': 3, 'learning_rate': 0.2876604512282171, 'batch_size': 11}
trial   7/60  val_loss=0.1085  epochs=100  {'hidden1': 16, 'hidden2': 4, 'learning_rate': 0.36785860778377877, 'batch_size': 26}
trial   8/60  val_loss=0.1419  epochs=1000  {'hidden1': 28, 'hidden2': 28, 'learn

In [11]:
# ---------------- Autoencoder Neural Network ----------------
print("Autoencoder Neural Network")
ae_space = {
    "n_encoding": ("int", 2, X.shape[1] - 1),                       
    "hidden2": ("int", 2, min(X.shape[1] * 2, 32)),
    "learning_rate": ("log_uniform", 0.001, 1.0),
    "batch_size": ("int", 8, max(8, X_cv.shape[0] // 10))             
}
ae_factory = lambda c, s: AutoencoderPredictor(
    X.shape[1], c["n_encoding"], c["hidden2"], y.shape[1],
    mode="classification", learning_rate=c["learning_rate"], seed=s)

best_ae, ae_trials = random_search(
    ae_factory, ae_space, X_cv, y_cv, X_val, y_val, yl_val, mode="classification",
    n_trials=60, max_epochs=1000, checkpoint_every=50, seed=seed,
    log_path="../results/tuning/house_votes_autoencoder.csv")
print("Best:", best_ae)

results = cross_validation_loop(
    X_cv, y_cv, yl_cv, lambda: ae_factory(best_ae, seed),
    epochs=best_ae["epochs"], batch_size=best_ae["batch_size"], seed=seed)
print(f"Mean Classification Error: {np.mean(results['error'])} \nMean Cross Entropy Loss: {np.mean(results['cross_entropy'])}")

Autoencoder Neural Network
trial   1/60  val_loss=0.1834  epochs=1000  {'n_encoding': 41, 'hidden2': 21, 'learning_rate': 0.006447036682716461, 'batch_size': 16}
trial   2/60  val_loss=0.2386  epochs=850  {'n_encoding': 3, 'hidden2': 4, 'learning_rate': 0.2753028565072983, 'batch_size': 8}
trial   3/60  val_loss=0.0969  epochs=600  {'n_encoding': 31, 'hidden2': 30, 'learning_rate': 0.06605524798050313, 'batch_size': 34}
trial   4/60  val_loss=0.3518  epochs=900  {'n_encoding': 35, 'hidden2': 21, 'learning_rate': 0.6385828802382952, 'batch_size': 22}
trial   5/60  val_loss=0.6527  epochs=1000  {'n_encoding': 14, 'hidden2': 27, 'learning_rate': 0.0010190969469382445, 'batch_size': 18}
trial   6/60  val_loss=0.1141  epochs=50  {'n_encoding': 41, 'hidden2': 19, 'learning_rate': 0.15451346766855226, 'batch_size': 8}
trial   7/60  val_loss=0.0944  epochs=50  {'n_encoding': 40, 'hidden2': 7, 'learning_rate': 0.3886303977238022, 'batch_size': 8}
trial   8/60  val_loss=0.1007  epochs=1000  {'n_

### Abalone Dataset

In [15]:
# Abalone Dataset
data, meta = load_processed("../data/processed/abalone_processed.csv")
X, y, y_labels = prepare_xy(data, meta)

# 20% tuning set, 80% for the 5x2 CV (same split for every model)
cv_idx, val_idx = holdout_split(len(X), None, 0.2, seed=seed)
X_cv, y_cv = X[cv_idx], y[cv_idx]
X_val, y_val = X[val_idx], y[val_idx]

In [13]:
# ---------------- Linear Regression ----------------
print("Linear Regression")
lin_space = {"learning_rate": ("log_uniform", 0.001, 1.0),
             "batch_size": ("int", 8, max(8, X_cv.shape[0] // 10))}
lin_factory = lambda c, s: LinearRegression(X.shape[1], c["learning_rate"], seed=s)

best_lin, lin_trials = random_search(
    lin_factory, lin_space, X_cv, y_cv, X_val, y_val, None, mode="regression",
    n_trials=30, max_epochs=1000, checkpoint_every=50, seed=seed,
    log_path="../results/tuning/abalone_linear.csv")
print("Best:", best_lin)

results = cross_validation_loop(
    X_cv, y_cv, None, lambda: lin_factory(best_lin, seed), mode="regression",
    epochs=best_lin["epochs"], batch_size=best_lin["batch_size"], seed=seed)
print(f"Mean Squared Error: {np.mean(results['mse'])}\n")

Linear Regression
trial   1/30  val_loss=0.4461  epochs=450  {'learning_rate': 0.081448869730761, 'batch_size': 175}
trial   2/30  val_loss=0.4601  epochs=1000  {'learning_rate': 0.0013271517125308802, 'batch_size': 96}
trial   3/30  val_loss=0.4587  epochs=1000  {'learning_rate': 0.0011209413928285786, 'batch_size': 65}
trial   4/30  val_loss=inf  epochs=nan  {'learning_rate': 0.5473520283788641, 'batch_size': 273} (diverged)
trial   5/30  val_loss=0.4473  epochs=750  {'learning_rate': 0.06605524798050313, 'batch_size': 325}
trial   6/30  val_loss=0.4476  epochs=800  {'learning_rate': 0.04274398161693902, 'batch_size': 246}
trial   7/30  val_loss=inf  epochs=nan  {'learning_rate': 0.6385828802382952, 'batch_size': 98} (diverged)
trial   8/30  val_loss=0.4663  epochs=1000  {'learning_rate': 0.0010190969469382445, 'batch_size': 274}
trial   9/30  val_loss=4906804641011551450308875723702099347743609085323708624458585002856607745090187135232246384268963069356808918214190695416494052667083

In [17]:
# ---------------- Neural Network ----------------
print("Neural Network")
nn_space = {"hidden1": ("int", 2, min(X.shape[1]*2, 32)), "hidden2": ("int", 2, min(X.shape[1]*2, 32)),
            "learning_rate": ("log_uniform", 0.001, 1.0), "batch_size": ("int", 8, max(8, X_cv.shape[0]//10))}
nn_factory = lambda c, s: NeuralNetwork(
    [X.shape[1], c["hidden1"], c["hidden2"], y.shape[1]],
    mode="regression", learning_rate=c["learning_rate"], seed=s)
best_nn, nn_trials = random_search(
    nn_factory, nn_space, X_cv, y_cv, X_val, y_val, None, mode="regression",
    n_trials=60, max_epochs=1000, checkpoint_every=50,
    constraint=lambda c: c["hidden2"] <= c["hidden1"], seed=seed,
    log_path="../results/tuning/abalone_nn.csv")
print("Best:", best_nn)

results = cross_validation_loop(
    X_cv, y_cv, None, lambda: nn_factory(best_nn, seed), mode="regression",
    epochs=best_nn["epochs"], batch_size=best_nn["batch_size"], seed=seed)
print(f"Mean Squared Error: {np.mean(results['mse'])}")

Neural Network
trial   1/60  val_loss=0.9330  epochs=950  {'hidden1': 18, 'hidden2': 14, 'learning_rate': 0.006447036682716461, 'batch_size': 108}
trial   2/60  val_loss=0.4069  epochs=1000  {'hidden1': 15, 'hidden2': 14, 'learning_rate': 0.6385828802382952, 'batch_size': 185}
trial   3/60  val_loss=0.4019  epochs=600  {'hidden1': 18, 'hidden2': 12, 'learning_rate': 0.15451346766855226, 'batch_size': 18}
trial   4/60  val_loss=0.4062  epochs=150  {'hidden1': 18, 'hidden2': 5, 'learning_rate': 0.3886303977238022, 'batch_size': 15}
trial   5/60  val_loss=0.5453  epochs=1000  {'hidden1': 12, 'hidden2': 3, 'learning_rate': 0.018537948170285826, 'batch_size': 106}
trial   6/60  val_loss=0.9325  epochs=150  {'hidden1': 9, 'hidden2': 2, 'learning_rate': 0.002359662167270631, 'batch_size': 10}
trial   7/60  val_loss=0.4084  epochs=1000  {'hidden1': 14, 'hidden2': 11, 'learning_rate': 0.07017062333532993, 'batch_size': 219}
trial   8/60  val_loss=0.4050  epochs=1000  {'hidden1': 16, 'hidden2': 

In [18]:
# ---------------- Autoencoder Neural Network ----------------
print("Autoencoder Neural Network")
ae_space = {
    "n_encoding": ("int", 2, X.shape[1] - 1),                       
    "hidden2": ("int", 2, min(X.shape[1] * 2, 32)),
    "learning_rate": ("log_uniform", 0.001, 1.0),
    "batch_size": ("int", 8, max(8, X_cv.shape[0] // 10))             
}
ae_factory = lambda c, s: AutoencoderPredictor(
    X.shape[1], c["n_encoding"], c["hidden2"], y.shape[1],
    mode="regression", learning_rate=c["learning_rate"], seed=s)

best_ae, ae_trials = random_search(
    ae_factory, ae_space, X_cv, y_cv, X_val, y_val, None, mode="regression",
    n_trials=60, max_epochs=1000, checkpoint_every=50, seed=seed,
    log_path="../results/tuning/abalone_autoencoder.csv")
print("Best:", best_ae)

results = cross_validation_loop(
    X_cv, y_cv, None, lambda: ae_factory(best_ae, seed),
    epochs=best_ae["epochs"], batch_size=best_ae["batch_size"], seed=seed, mode="regression")
print(f"Mean Squared Error: {np.mean(results['mse'])}")

Autoencoder Neural Network
trial   1/60  val_loss=0.5927  epochs=1000  {'n_encoding': 8, 'hidden2': 14, 'learning_rate': 0.006447036682716461, 'batch_size': 108}
trial   2/60  val_loss=0.5970  epochs=700  {'n_encoding': 2, 'hidden2': 3, 'learning_rate': 0.2753028565072983, 'batch_size': 13}
trial   3/60  val_loss=0.5001  epochs=1000  {'n_encoding': 7, 'hidden2': 19, 'learning_rate': 0.06605524798050313, 'batch_size': 325}
trial   4/60  val_loss=0.4473  epochs=800  {'n_encoding': 7, 'hidden2': 14, 'learning_rate': 0.6385828802382952, 'batch_size': 185}
trial   5/60  val_loss=0.9173  epochs=1000  {'n_encoding': 4, 'hidden2': 17, 'learning_rate': 0.0010190969469382445, 'batch_size': 136}
trial   6/60  val_loss=0.4326  epochs=900  {'n_encoding': 8, 'hidden2': 12, 'learning_rate': 0.15451346766855226, 'batch_size': 18}
trial   7/60  val_loss=0.4551  epochs=450  {'n_encoding': 8, 'hidden2': 5, 'learning_rate': 0.3886303977238022, 'batch_size': 15}
trial   8/60  val_loss=0.4902  epochs=1000  

### Computer Hardware Dataset

In [19]:
# Computer Hardware Dataset
data, meta = load_processed("../data/processed/computer_hardware_processed.csv")
X, y, y_labels = prepare_xy(data, meta)

# 20% tuning set, 80% for the 5x2 CV (same split for every model)
cv_idx, val_idx = holdout_split(len(X), None, 0.2, seed=seed)
X_cv, y_cv = X[cv_idx], y[cv_idx]
X_val, y_val = X[val_idx], y[val_idx]

In [20]:
# ---------------- Linear Regression ----------------
print("Linear Regression")
lin_space = {"learning_rate": ("log_uniform", 0.001, 1.0),
             "batch_size": ("int", 8, max(8, X_cv.shape[0] // 10))}
lin_factory = lambda c, s: LinearRegression(X.shape[1], c["learning_rate"], seed=s)

best_lin, lin_trials = random_search(
    lin_factory, lin_space, X_cv, y_cv, X_val, y_val, None, mode="regression",
    n_trials=30, max_epochs=1000, checkpoint_every=50, seed=seed,
    log_path="../results/tuning/computer_hardware_linear.csv")
print("Best:", best_lin)

results = cross_validation_loop(
    X_cv, y_cv, None, lambda: lin_factory(best_lin, seed), mode="regression",
    epochs=best_lin["epochs"], batch_size=best_lin["batch_size"], seed=seed)
print(f"Mean Squared Error: {np.mean(results['mse'])}\n")

Linear Regression
trial   1/30  val_loss=0.1336  epochs=450  {'learning_rate': 0.081448869730761, 'batch_size': 12}
trial   2/30  val_loss=0.1346  epochs=50  {'learning_rate': 0.0013271517125308802, 'batch_size': 10}
trial   3/30  val_loss=0.1339  epochs=50  {'learning_rate': 0.0011209413928285786, 'batch_size': 9}
trial   4/30  val_loss=140.5046  epochs=500  {'learning_rate': 0.5473520283788641, 'batch_size': 15}
trial   5/30  val_loss=0.1295  epochs=350  {'learning_rate': 0.06605524798050313, 'batch_size': 16}
trial   6/30  val_loss=0.1447  epochs=250  {'learning_rate': 0.04274398161693902, 'batch_size': 14}
trial   7/30  val_loss=31098030437343354847771489598487323570562671225933432657995028758528.0000  epochs=50  {'learning_rate': 0.6385828802382952, 'batch_size': 10} (diverged)
trial   8/30  val_loss=0.1160  epochs=50  {'learning_rate': 0.0010190969469382445, 'batch_size': 15}
trial   9/30  val_loss=0.2538  epochs=450  {'learning_rate': 0.37343311708676946, 'batch_size': 12}
trial

In [21]:
# ---------------- Neural Network ----------------
print("Neural Network")
nn_space = {"hidden1": ("int", 2, min(X.shape[1]*2, 32)), "hidden2": ("int", 2, min(X.shape[1]*2, 32)),
            "learning_rate": ("log_uniform", 0.001, 1.0), "batch_size": ("int", 8, max(8, X_cv.shape[0]//10))}
nn_factory = lambda c, s: NeuralNetwork(
    [X.shape[1], c["hidden1"], c["hidden2"], y.shape[1]],
    mode="regression", learning_rate=c["learning_rate"], seed=s)

best_nn, nn_trials = random_search(
    nn_factory, nn_space, X_cv, y_cv, X_val, y_val, None, mode="regression",
    n_trials=60, max_epochs=1000, checkpoint_every=50,
    constraint=lambda c: c["hidden2"] <= c["hidden1"], seed=seed,
    log_path="../results/tuning/computer_hardware_nn.csv")
print("Best:", best_nn)

results = cross_validation_loop(
    X_cv, y_cv, None, lambda: nn_factory(best_nn, seed), mode="regression",
    epochs=best_nn["epochs"], batch_size=best_nn["batch_size"], seed=seed)
print(f"Mean Squared Error: {np.mean(results['mse'])}")

Neural Network
trial   1/60  val_loss=0.3488  epochs=650  {'hidden1': 11, 'hidden2': 9, 'learning_rate': 0.006447036682716461, 'batch_size': 10}
trial   2/60  val_loss=0.1097  epochs=100  {'hidden1': 2, 'hidden2': 2, 'learning_rate': 0.2753028565072983, 'batch_size': 8}
trial   3/60  val_loss=0.0759  epochs=600  {'hidden1': 10, 'hidden2': 8, 'learning_rate': 0.6385828802382952, 'batch_size': 12}
trial   4/60  val_loss=0.0791  epochs=1000  {'hidden1': 11, 'hidden2': 8, 'learning_rate': 0.15451346766855226, 'batch_size': 8}
trial   5/60  val_loss=0.0622  epochs=850  {'hidden1': 11, 'hidden2': 3, 'learning_rate': 0.3886303977238022, 'batch_size': 8}
trial   6/60  val_loss=0.3472  epochs=50  {'hidden1': 7, 'hidden2': 2, 'learning_rate': 0.018537948170285826, 'batch_size': 10}
trial   7/60  val_loss=0.3512  epochs=50  {'hidden1': 6, 'hidden2': 2, 'learning_rate': 0.002359662167270631, 'batch_size': 8}
trial   8/60  val_loss=0.0719  epochs=700  {'hidden1': 9, 'hidden2': 7, 'learning_rate': 0

In [22]:
# ---------------- Autoencoder Neural Network ----------------
print("Autoencoder Neural Network")
ae_space = {
    "n_encoding": ("int", 2, X.shape[1] - 1),                       
    "hidden2": ("int", 2, min(X.shape[1] * 2, 32)),
    "learning_rate": ("log_uniform", 0.001, 1.0),
    "batch_size": ("int", 8, max(8, X_cv.shape[0] // 10))             
}
ae_factory = lambda c, s: AutoencoderPredictor(
    X.shape[1], c["n_encoding"], c["hidden2"], y.shape[1],
    mode="regression", learning_rate=c["learning_rate"], seed=s)

best_ae, ae_trials = random_search(
    ae_factory, ae_space, X_cv, y_cv, X_val, y_val, None, mode="regression",
    n_trials=60, max_epochs=1000, checkpoint_every=50, seed=seed,
    log_path="../results/tuning/computer_hardware_autoencoder.csv")
print("Best:", best_ae)

results = cross_validation_loop(
    X_cv, y_cv, None, lambda: ae_factory(best_ae, seed),
    epochs=best_ae["epochs"], batch_size=best_ae["batch_size"], seed=seed, mode="regression")
print(f"Mean Squared Error: {np.mean(results['mse'])}")

Autoencoder Neural Network
trial   1/60  val_loss=0.1076  epochs=650  {'n_encoding': 5, 'hidden2': 9, 'learning_rate': 0.006447036682716461, 'batch_size': 10}
trial   2/60  val_loss=0.0825  epochs=950  {'n_encoding': 2, 'hidden2': 2, 'learning_rate': 0.2753028565072983, 'batch_size': 8}
trial   3/60  val_loss=0.1091  epochs=150  {'n_encoding': 4, 'hidden2': 12, 'learning_rate': 0.06605524798050313, 'batch_size': 16}
trial   4/60  val_loss=0.0894  epochs=350  {'n_encoding': 4, 'hidden2': 8, 'learning_rate': 0.6385828802382952, 'batch_size': 12}
trial   5/60  val_loss=0.3478  epochs=900  {'n_encoding': 3, 'hidden2': 10, 'learning_rate': 0.0010190969469382445, 'batch_size': 11}
trial   6/60  val_loss=0.0553  epochs=800  {'n_encoding': 5, 'hidden2': 8, 'learning_rate': 0.15451346766855226, 'batch_size': 8}
trial   7/60  val_loss=0.0799  epochs=200  {'n_encoding': 5, 'hidden2': 3, 'learning_rate': 0.3886303977238022, 'batch_size': 8}
trial   8/60  val_loss=0.1034  epochs=400  {'n_encoding':

### Forest Fires Dataset

In [23]:
# Forest Fires Dataset
data, meta = load_processed("../data/processed/forest_fires_processed.csv")
X, y, y_labels = prepare_xy(data, meta)

# 20% tuning set, 80% for the 5x2 CV (same split for every model)
cv_idx, val_idx = holdout_split(len(X), None, 0.2, seed=seed)
X_cv, y_cv = X[cv_idx], y[cv_idx]
X_val, y_val = X[val_idx], y[val_idx]

In [24]:
# ---------------- Linear Regression ----------------
print("Linear Regression")
lin_space = {"learning_rate": ("log_uniform", 0.001, 1.0),
             "batch_size": ("int", 8, max(8, X_cv.shape[0] // 10))}
lin_factory = lambda c, s: LinearRegression(X.shape[1], c["learning_rate"], seed=s)

best_lin, lin_trials = random_search(
    lin_factory, lin_space, X_cv, y_cv, X_val, y_val, None, mode="regression",
    n_trials=30, max_epochs=1000, checkpoint_every=50, seed=seed,
    log_path="../results/tuning/forest_fires_linear.csv")
print("Best:", best_lin)

results = cross_validation_loop(
    X_cv, y_cv, None, lambda: lin_factory(best_lin, seed), mode="regression",
    epochs=best_lin["epochs"], batch_size=best_lin["batch_size"], seed=seed)
print(f"Mean Squared Error: {np.mean(results['mse'])}\n")

Linear Regression
trial   1/30  val_loss=0.8827  epochs=800  {'learning_rate': 0.081448869730761, 'batch_size': 25}
trial   2/30  val_loss=0.9306  epochs=850  {'learning_rate': 0.0013271517125308802, 'batch_size': 17}
trial   3/30  val_loss=0.9312  epochs=950  {'learning_rate': 0.0011209413928285786, 'batch_size': 13}
trial   4/30  val_loss=56797951276846696116366316813250751560792670208.0000  epochs=50  {'learning_rate': 0.5473520283788641, 'batch_size': 35} (diverged)
trial   5/30  val_loss=0.8634  epochs=300  {'learning_rate': 0.06605524798050313, 'batch_size': 41}
trial   6/30  val_loss=0.9074  epochs=400  {'learning_rate': 0.04274398161693902, 'batch_size': 32}
trial   7/30  val_loss=681854122854529248600497316006007990953703879431735697356659340192548754155628064632982730992282278884941253962684482938213360262206609342173644380858132448118566101679215455234285483110688280614613290087950288529763971636163676230004306154881856459812670275584.0000  epochs=50  {'learning_rate': 0.638

In [25]:
# ---------------- Neural Network ----------------
print("Neural Network")
nn_space = {"hidden1": ("int", 2, min(X.shape[1]*2, 32)), "hidden2": ("int", 2, min(X.shape[1]*2, 32)),
            "learning_rate": ("log_uniform", 0.001, 1.0), "batch_size": ("int", 8, max(8, X_cv.shape[0]//10))}
nn_factory = lambda c, s: NeuralNetwork(
    [X.shape[1], c["hidden1"], c["hidden2"], y.shape[1]],
    mode="regression", learning_rate=c["learning_rate"], seed=s)

best_nn, nn_trials = random_search(
    nn_factory, nn_space, X_cv, y_cv, X_val, y_val, None, mode="regression",
    n_trials=60, max_epochs=1000, checkpoint_every=50,
    constraint=lambda c: c["hidden2"] <= c["hidden1"], seed=seed,
    log_path="../results/tuning/forest_fires_nn.csv")
print("Best:", best_nn)

results = cross_validation_loop(
    X_cv, y_cv, None, lambda: nn_factory(best_nn, seed), mode="regression",
    epochs=best_nn["epochs"], batch_size=best_nn["batch_size"], seed=seed)
print(f"Mean Squared Error: {np.mean(results['mse'])}")

Neural Network
trial   1/60  val_loss=0.8864  epochs=700  {'hidden1': 24, 'hidden2': 19, 'learning_rate': 0.006447036682716461, 'batch_size': 18}
trial   2/60  val_loss=0.9352  epochs=200  {'hidden1': 21, 'hidden2': 19, 'learning_rate': 0.6385828802382952, 'batch_size': 26}
trial   3/60  val_loss=0.8987  epochs=50  {'hidden1': 25, 'hidden2': 16, 'learning_rate': 0.15451346766855226, 'batch_size': 9}
trial   4/60  val_loss=0.9580  epochs=100  {'hidden1': 24, 'hidden2': 6, 'learning_rate': 0.3886303977238022, 'batch_size': 8}
trial   5/60  val_loss=0.8852  epochs=150  {'hidden1': 16, 'hidden2': 4, 'learning_rate': 0.018537948170285826, 'batch_size': 18}
trial   6/60  val_loss=0.8881  epochs=550  {'hidden1': 12, 'hidden2': 2, 'learning_rate': 0.002359662167270631, 'batch_size': 8}
trial   7/60  val_loss=0.8792  epochs=250  {'hidden1': 20, 'hidden2': 16, 'learning_rate': 0.07017062333532993, 'batch_size': 30}
trial   8/60  val_loss=0.9597  epochs=50  {'hidden1': 22, 'hidden2': 12, 'learnin

In [26]:
# ---------------- Autoencoder Neural Network ----------------
print("Autoencoder Neural Network")
ae_space = {
    "n_encoding": ("int", 2, X.shape[1] - 1),                       
    "hidden2": ("int", 2, min(X.shape[1] * 2, 32)),
    "learning_rate": ("log_uniform", 0.001, 1.0),
    "batch_size": ("int", 8, max(8, X_cv.shape[0] // 10))             
}
ae_factory = lambda c, s: AutoencoderPredictor(
    X.shape[1], c["n_encoding"], c["hidden2"], y.shape[1],
    mode="regression", learning_rate=c["learning_rate"], seed=s)

best_ae, ae_trials = random_search(
    ae_factory, ae_space, X_cv, y_cv, X_val, y_val, None, mode="regression",
    n_trials=60, max_epochs=1000, checkpoint_every=50, seed=seed,
    log_path="../results/tuning/forest_fires_autoencoder.csv")
print("Best:", best_ae)

results = cross_validation_loop(
    X_cv, y_cv, None, lambda: ae_factory(best_ae, seed),
    epochs=best_ae["epochs"], batch_size=best_ae["batch_size"], seed=seed, mode="regression")
print(f"Mean Squared Error: {np.mean(results['mse'])}")

Autoencoder Neural Network
trial   1/60  val_loss=0.8869  epochs=300  {'n_encoding': 12, 'hidden2': 19, 'learning_rate': 0.006447036682716461, 'batch_size': 18}
trial   2/60  val_loss=0.9222  epochs=850  {'n_encoding': 2, 'hidden2': 4, 'learning_rate': 0.2753028565072983, 'batch_size': 8}
trial   3/60  val_loss=0.8779  epochs=200  {'n_encoding': 9, 'hidden2': 26, 'learning_rate': 0.06605524798050313, 'batch_size': 41}
trial   4/60  val_loss=0.9079  epochs=100  {'n_encoding': 10, 'hidden2': 19, 'learning_rate': 0.6385828802382952, 'batch_size': 26}
trial   5/60  val_loss=0.8882  epochs=200  {'n_encoding': 5, 'hidden2': 24, 'learning_rate': 0.0010190969469382445, 'batch_size': 21}
trial   6/60  val_loss=0.9151  epochs=50  {'n_encoding': 12, 'hidden2': 16, 'learning_rate': 0.15451346766855226, 'batch_size': 9}
trial   7/60  val_loss=0.9536  epochs=50  {'n_encoding': 12, 'hidden2': 6, 'learning_rate': 0.3886303977238022, 'batch_size': 8}
trial   8/60  val_loss=0.8851  epochs=500  {'n_encod